#  LLM Flake Analysis Automation: Step 1

---

### Goal: Automate flake analysis using an LLM and save the results to a database

**By: Sanjit Masanam (2025, UCSB)**

Notebook inpired by Glenn Parham's ([source](https://githubtocolab.com/deptofdefense/LLMs-at-DoD/blob/main/tutorials/Open_Source_LLMs_Getting_Started.ipynb))

## Getting Started

TLDR: Use T4/A100 GPU and click "yankowitzlab@gmail.com", then "Continue" 2x when prompted to give Colab access to the Drive.

In this colab notebook, we'll want to use a **T4** GPU (A100/L4 will also work).  You may switch to this by going to "Runtime" then "Change Runtime Type". This is to ensure it doesn't take forever for the LLM to run.

Here we'll do some setup actions such as mounting the Lab Google Drive to access LLM models + flake images. When prompted, click "Continue" twice to give Google Colab access to the Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
main_path = '/content/drive/Shareddrives/Yankowitz Lab/'

Mounted at /content/drive


To get around annoying text-wrapping issues, run this cell.


In [ ]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

We'll need to install the following requirements:
- HuggingFace
- Llama-cpp-python with CUDA support
  - Building the wheel for this library takes forever so instead we download a prebuilt wheel for CUDA 12.4. This doesn't match the CUDA version on Colab (12.5) but it seems to work regardless.

In [ ]:
## Installing huggingface and llama-cpp-python with support for CUDA 12.4
!pip install huggingface-hub==0.17.1 -q
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.8/294.8 kB 23.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.0 requires huggingface-hub>=0.20.0, but you have huggingface-hub 0.17.1 which is incompatible.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.17.1 which is incompatible.
gradio-client 1.11.1 requires huggingface-hub>=0.19.3, but you have huggingface-hub 0.17.1 which is incompatible.
diffusers 0.34.0 requires huggingface-hub>=0.27.0, but you have huggingface-hub 0.17.1 which is incompatible.
accelerate 1.10.0 requires huggingface_hub>=0.21.0, but you have huggingface-hub 0.17.1 which is incompatible.
transformers 4.55.2 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 0.17.1 which is incompatible.
peft 0.17.0 requires huggingface_hub>=0.25.0, but you have hug

## Model Setup

We'll need to download the model weights from HuggingFace (in case they aren't already downloaded to the Drive).

We'll be using the [Llama-4-Scout-17B-16E-Instruct-GGUF](https://huggingface.co/unsloth/Llama-4-Scout-17B-16E-Instruct-GGUF) model. If you ever want to experiment with a different model, feel free to try one of these: https://huggingface.co/models?pipeline_tag=image-text-to-text&sort=trending&search=GGUF

Note: The model you select **must** be of type "GGUF"

GGUF is...
- binary file format for storing models for inference
- designed for fast loading and saving of models
- easy to use (with a few lines of code)
- mmap (memory mapping) compatibility: models can be loaded using mmap for fast loading and saving.

In [ ]:
selected_llm = 'Llama-4-Scout-17B-16E-Instruct-GGUF'

# Feel free to add more options to this model_dict
model_dict = {
    "Llama-4-Scout-17B-16E-Instruct-GGUF":{"HF_REPO_NAME":"unsloth/Llama-4-Scout-17B-16E-Instruct-GGUF","HF_MODEL_NAME":"Llama-4-Scout-17B-16E-Instruct-UD-IQ2_M.gguf","HF_CLIP_MODEL_NAME":"mmproj-F16.gguf"}
             }

In [ ]:
import os
from huggingface_hub import hf_hub_download

HF_REPO_NAME = model_dict[selected_llm]['HF_REPO_NAME']
HF_MODEL_NAME = model_dict[selected_llm]['HF_MODEL_NAME']
HF_CLIP_MODEL_NAME = model_dict[selected_llm]['HF_CLIP_MODEL_NAME']
LOCAL_DIR_NAME = f"/content/drive/Shareddrives/Yankowitz Lab/LLM/{selected_llm}/"

os.makedirs(LOCAL_DIR_NAME, exist_ok=True)

if os.path.exists(LOCAL_DIR_NAME+HF_MODEL_NAME) == False:
  hf_hub_download(repo_id=HF_REPO_NAME, filename=HF_MODEL_NAME, local_dir=LOCAL_DIR_NAME)

if os.path.exists(LOCAL_DIR_NAME+HF_CLIP_MODEL_NAME) == False:
  hf_hub_download(repo_id=HF_REPO_NAME, filename=HF_CLIP_MODEL_NAME, local_dir=LOCAL_DIR_NAME)

Now, let's initialize the "Llama" framework.

Note: Naming is a bit messy.  Llama-cpp was named after Meta's open-source "*Llama*" LLMs.  The framework was built to make it easy to locally run & program with this LLM.  However, now, the framework as been abstracted and modified to work with ***any*** open-source text-generation LLM, as long as it is in the GGUF model file type.

In [ ]:
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Llama3VisionAlphaChatHandler
import base64
import os

chat_handler = Llama3VisionAlphaChatHandler(
  clip_model_path=LOCAL_DIR_NAME+'/'+HF_CLIP_MODEL_NAME
)

llm = Llama(
    model_path=LOCAL_DIR_NAME+'/'+HF_MODEL_NAME,
    chat_handler=chat_handler,
    n_ctx=4096, # n_ctx should be increased to accommodate the image embedding
    n_threads=2,
    n_gpu_layers=-1,
    n_batch=512,
    verbose=True
)

def image_to_base64_data_uri(file_path):
    with open(file_path, "rb") as img_file:
        base64_data = base64.b64encode(img_file.read()).decode('utf-8')
        return f"data:image/png;base64,{base64_data}"

## The Fun Part

Hopefully you got here without too much trouble. Now, we'll need a bit of information from you and then the LLM can get to work! Please follow the prompts in the cell output.

**Status (Aug. 2025):** Code runs in a reasonable amount of time with the A100 GPU but responses aren't of high enough quality to be useful. If all else fails, time could solve this issue as models improve.

In [ ]:
### Initialize dataframe
flake_df = pd.read_csv(LOCAL_DIR_NAME+'../flake_df.csv')
print(flake_df.head)

### Ask for necessary info on what flakes to analyze, then set request_type
print("Hello! This is LLMatt at your service (most of the time...)! Please enter the full path to flake or directory of the flakes you would like me to analyze:")
bool = True
while bool == True:
    path = input()
    if os.path.exists(path) == False:
        print("Sorry, that path does not lead anywhere. Please try again:")
    else:
        bool = False

print("What material is this flake(s) made of:")
material = input()
print("Thanks, I'll work on analyzing now!")

if os.path.isfile(path) == True:
    request_type = 'single'
elif os.path.isdir(path) == True:
    request_type = 'multi'

if request_type == 'single':
  data_uri = image_to_base64_data_uri(path)
  response = llm.create_chat_completion(
    messages = [
        {"role": "system", "content": "You are an assistant who analyzes 2D vdW material flakes."},
        {"role": "user",
            "content": [
                {"type" : "text", "text": f"This is a 2D Van der Waals flake made of {material}. Please approximate the number of layers this flake is made of, and provide a rating from 1 to 10 in these categories: shape, size, defects, usability. Please output in this format: layer_num, shape_rating, size_rating, defect_rating, usability_rating. In addition, please provide a few tags that describe the main features of this flake (width, height, shape, defects, etc.) seperated by |."},
                {"type": "image_url", "image_url": {"url": data_uri }}
            ]}])
    layer_num, shape_rating, size_rating, defect_rating, usability_rating, tags = response["choices"][0]["text"].split(", ")
    flake_df.append({'flake_filepath': path, 'used': 0, 'material': material, 'layer_num': layer_num, 'shape_rating': shape_rating, 'size_rating': size_rating, 'defect_rating': defect_rating, 'usability_rating': usability_rating, 'tags': tags})
    print("Saved to df!")
    print(response)

if request_type == 'multi':
    for filename in os.listdir(path):
      data_uri = image_to_base64_data_uri(path+filename)
      response = llm.create_chat_completion(
        messages = [
            {"role": "system", "content": "You are an assistant who analyzes 2D vdW material flakes."},
            {"role": "user",
                "content": [
                    {"type" : "text", "text": f"This is a 2D Van der Waals flake made of {material}. Please approximate the number of layers this flake is made of, and provide a rating from 1 to 10 in these categories: shape, size, defects, usability. Please output in this format: layer_num, shape_rating, size_rating, defect_rating, usability_rating. In addition, please provide a few tags that describe the main features of this flake (width, height, shape, defects, etc.) seperated by |."},
                    {"type": "image_url", "image_url": {"url": data_uri }}
                ]}])
      layer_num, shape_rating, size_rating, defect_rating, usability_rating, tags = response["choices"][0]["text"].split(", ")
      flake_df.append({'flake_filepath': path, 'used': 0, 'material': material, 'layer_num': layer_num, 'shape_rating': shape_rating, 'size_rating': size_rating, 'defect_rating': defect_rating, 'usability_rating': usability_rating, 'tags': tags})
      print("Saved to df!")
      print(response)

### Save df to csv
flake_df.to_csv(main_path+'LLM/flake_df.csv')